**Youtube template**

*   https://www.youtube.com/watch?v=hKVhRA9kfeM&t=2901s
*   https://github.com/alejandro-ao/agents-no-framework/blob/master/agent-w-loop.ipynb


#### dependencies

In [ ]:
!pip -q install langchain_openai openai python-dotenv langchain langchain_community pypdf sentence-transformers PyPDF2 pycryptodome tiktoken faiss-gpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/294.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 k

In [ ]:
%%writefile .env
OPENAI_API_KEY=""


Overwriting .env


In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

#### test openai connection

In [ ]:
import os
from openai import OpenAI
import re
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


messages = [
        {"role": "system", "content": 'You are a sport fanatic'},
        {"role": "user", "content": 'how many superbowls has Rams won'},
    ]
response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0
    )
response_message = response.choices[0].message.content
print(response_message )

As of October 2023, the Los Angeles Rams have won two Super Bowls. Their first victory came in Super Bowl XXXIV during the 1999 season when they were based in St. Louis. They won their second Super Bowl in the 2021 season, capturing the title in Super Bowl LVI.


#### build RAG search tool

In [ ]:
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain import FAISS


In [ ]:
pdf_reader = PdfReader('/content/naf23.pdf')
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

In [ ]:
def process_text(text, chunk_size=1000, chunk_overlap=100):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    embeddings = OpenAIEmbeddings()
    knowledgeBase = FAISS.from_texts(chunks, embeddings)
    return knowledgeBase

knowledgeBase = process_text(text)

In [ ]:
def search(query):
    return knowledgeBase.similarity_search(query, k=1)

In [ ]:
search('total revenue')

[Document(metadata={}, page_content='Total Revenue 979,651           -                        979,651           \nNet Assets Released from Restrictions 975,954           (975,954)          -                        \nTotal Support and Revenue 3,546,026       638,761           4,184,787       \nExpenses\nProgram services\nResearch 1,574,354       -                        1,574,354       \nEducation and service 1,146,980       -                        1,146,980       \nDrug Development Collaborative 1,188,427       -                        1,188,427       \nTotal Program Services 3,909,761       -                        3,909,761       \nSupporting services\nManagement and general 538,079           -                        538,079           \nFundraising 548,618           -                        548,618           \nTotal Supporting Services 1,086,697       -                        1,086,697       \nTotal Expenses 4,996,458       -                        4,996,458')]

#### building ReAct agent and test it step by step

In [ ]:
class Agent:
    def __init__(self, client=client, system_prompt: str = "") -> None:
        self.client = client # the llm
        self.system_prompt = system_prompt
        self.messages: list = []
        if self.system_prompt:
            self.messages.append({"role": "system", "content": self.system_prompt})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=self.messages,
            temperature=0
        )
        return completion.choices[0].message.content

In [ ]:
system_prompt = """
you are a financial expert and an underwriter working for commerical banking of a bank.
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

search:
e.g. search: total income
Runs a serach function in the knowledge base and return relevant information as chunks of text.


Example session:

Question: analyze financial performance of Apple Inc and potential drivers?
Thought: to analyse finanial performance, I need to know Apple's total revenue, so I need to search its total revenue
Action: search: Apple Inc total revenue
PAUSE

You will be called again with this:

Observation: Document(metadata={}, page_content='Total Revenue 979,651...'

Thought: to know the potential drivers, I need to know the buisness composition of Apple, so I need to search its buisness composition
Action: search: Apple Inc buisness composition
PAUSE

You will be called again with this:

Observation: Document(metadata={}, page_content='Apple's business composition includes iPhone: Major revenue driver. Mac: Laptops and desktops...'

If you have the answer, output it as the Answer.

Answer: Apple's financial performance is strong with a total revenue of 979,651, which is drived by iPhone sales.

Now it's your turn:
""".strip()

In [ ]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:
class ReactExcecutor:
    def __init__(self, client, tools, system_prompt, max_iterations=10):

        self.client = client
        self.tools = tools
        self.system_prompt = system_prompt
        self.max_iterations = max_iterations

    def run(self, query):
        agent = Agent(client = self.client, system_prompt=self.system_prompt)
        next_prompt = query
        i = 0
        while i < self.max_iterations:
            i += 1
            result = agent(next_prompt)
            print(result)

            if "PAUSE" in result and "Action" in result:
                action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
                chosen_tool = action[0][0]
                arg = action[0][1]

                if chosen_tool in self.tools:
                    result_tool = eval(f"{chosen_tool}('{arg}')")
                    next_prompt = f"Observation: {result_tool}"

                else:
                    next_prompt = "Observation: Tool not found"

                print(next_prompt)
                continue

            if "Answer" in result:
                break

In [ ]:
reactExcecutor = ReactExcecutor(client=client, tools=["search"], system_prompt=system_prompt)

In [ ]:
reactExcecutor.run(query="what is total revenue in 2023")

Thought: To provide the total revenue for 2023, I need to search for the most recent financial data or reports that include this information.
Action: search: total revenue 2023
PAUSE
Observation: [Document(metadata={}, page_content='Total Revenue 979,651           -                        979,651           \nNet Assets Released from Restrictions 975,954           (975,954)          -                        \nTotal Support and Revenue 3,546,026       638,761           4,184,787       \nExpenses\nProgram services\nResearch 1,574,354       -                        1,574,354       \nEducation and service 1,146,980       -                        1,146,980       \nDrug Development Collaborative 1,188,427       -                        1,188,427       \nTotal Program Services 3,909,761       -                        3,909,761       \nSupporting services\nManagement and general 538,079           -                        538,079           \nFundraising 548,618           -                       

In [ ]:
reactExcecutor.run(query="Mary bought a $1k computer for the company, can it be considered as an equipment")

Thought: To determine if the $1,000 computer can be considered as equipment, I need to understand the accounting definition of equipment and how it is typically classified in financial statements. I will search for the definition and classification criteria of equipment in accounting. 
Action: search: accounting definition of equipment classification criteria
PAUSE
Observation: [Document(metadata={}, page_content='Note 1:  Summary of Significant Accounting Policies (Continued)  \n \nJ. Property and Equipment  \n \nProperty and equipment are capitalized at cost. Depreciation is provided for using the straight -line method over the \nestimated useful life of the asset. The Foundation considers items with a cost greater than $2,500 and a useful life \ngreater than one year to be property and equipment. Items below $2,500 are expensed when purchased.  \n \nK. Functional Expenses  \n \nThe costs of providing program and other activities have been summarized on a functional basis in the stat

In [ ]:
query ="How did deferred revenue change between 2022 and 2023, and what drove this change?"
reactExcecutor.run(query=query)

Thought: To understand the change in deferred revenue between 2022 and 2023, I need to find the deferred revenue figures for both years. Then, I need to identify the factors that contributed to this change. I will start by searching for the deferred revenue in 2022.
Action: search: deferred revenue 2022
PAUSE
Observation: [Document(metadata={}, page_content='agreements are signed annually. Earned income received in advance of the collaborative period is reported as deferred \nrevenue. In the case of conference income, revenue is recognized at the time the event is held, which is at a point in time. \nConference income received in advance of the event is reported as deferred revenue.  \n \n• Performance Obligations  \n \nThe performance obligation related to Drug D evelopment Collaborative is satisfied over the collaborative period; \ntherefore, the Foundation  recognizes revenue over a period of time on a monthly basis. The performance \nobligation related to the conference income is s

In [ ]:
query="What caused the significant change in net assets with donor restrictions between 2022 and 2023?"
reactExcecutor.run(query=query)

Thought: To determine the cause of the significant change in net assets with donor restrictions between 2022 and 2023, I need to find information on the financial statements or reports that detail these changes. This may include donations received, changes in donor restrictions, or expenditures related to these assets. I will start by searching for the financial statements or reports for the years 2022 and 2023.
Action: search: net assets with donor restrictions 2022 2023 financial statements
PAUSE
Observation: [Document(metadata={}, page_content="With donor restriction 2,680,567       2,041,806       \nTotal Net Assets 3,614,946       4,426,617       \nTotal Liabilities and Net Assets 4,116,826$     5,008,595$      National Ataxia Foundation\nStatements of Financial Position\nDecember 31, 2023 and 2022\nSee Independent Auditor's Report and Notes to the Financial Statements.\n6Without Donor With Donor\nRestriction Restriction Total\nSupport and Revenue\nSupport\nContributions, memorial

In [ ]:
query="What was the operating lease liability at the end of 2023, and how does the lease structure affect NAF’s financial obligations?"
reactExcecutor.run(query=query)

Thought: To answer this question, I need to find the operating lease liability for NAF at the end of 2023. I will search for this information first.
Action: search: NAF operating lease liability end of 2023
PAUSE
Observation: [Document(metadata={}, page_content='Weighted-average discount rate for operating leases 2.333%\nFuture minimum payments for leases are as follows:\nYear Ended December 31, Amount\n2024 33,693 $          \n2025 25,552             \nTotal undiscounted cash flows 59,245             \nLess: present value discount (1,142)              \nTotal Lease Liabilities 58,103 $          \n \n19 \n National Ataxia Foundation  \nNotes to the Financial Statements  \nDecember 31, 2023  and 2022  \n \nNote 6: Net Assets with Donor Restrictions  \n \nNet assets with donor restrictions were available for the following purposes at December 31:  \n \n2023 2022\nSCA2 Fund 880,250 $        500,000 $        \nMR Imaging Study Fund 612,044           -                        \nGeneral Resea